In [1]:
import re
import pandas as pd
import os

# ==========================================
# 1. CONFIGURATION
# ==========================================
INPUT_CSV_PATH = '/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/train_labeled.csv'
OUTPUT_CSV_PATH = '/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/alsei/Thesis_IT_TicketClassification/data/nuuday_distillation_ready.csv'

# ==========================================
# 2. BULLETPROOF PARSING LOGIC
# ==========================================
def parse_raw_data(row):
    raw_text = str(row['text'])
    ticket_number = str(row['number']).strip()
    
    clean_tag = str(row['label']).strip() 

    desc_match = re.search(
        r"description:\s*(.*?)(?=(?:[\s,]*reasoning:\s*(?:Tag:|\())|\nReasoning:)", 
        raw_text, 
        re.DOTALL | re.IGNORECASE
    )
    clean_desc = desc_match.group(1).strip() if desc_match else "N/A"
    
    final_text = f"description: {ticket_number} {clean_desc}"
    
    reas_match = re.search(
        r"\nReasoning:\s*(.*?)(?=\n(?:Reasoning_Confidence|Label_Confidence|Confidence|scientific_confidence):)", 
        raw_text, 
        re.DOTALL | re.IGNORECASE
    )
    clean_reas = reas_match.group(1).strip() if reas_match else "N/A"
    
    s_match = re.search(r"scientific_confidence:\s*([0-9.]+)", raw_text, re.IGNORECASE)
    s_star = float(s_match.group(1)) if s_match else 0.0
    
    # strictly Reasoning + Tag. No confidence.
    composite_label = (
        f"Reasoning: {clean_reas}\n"
        f"Tag: {clean_tag}"
    )
    
    # Return a 3-item series
    return pd.Series([final_text, composite_label, s_star])

# ==========================================
# 3. EXECUTION 
# ==========================================
print(f"Loading raw dataset: {INPUT_CSV_PATH}")
df = pd.read_csv(INPUT_CSV_PATH)

print("Executing bulletproof formatting (3 columns)...")
# Map the output to 3 columns
df[['text', 'label', 's_star']] = df.apply(parse_raw_data, axis=1)

# Save all 3 columns
final_df = df[['text', 'label', 's_star']].copy()
final_df.to_csv(OUTPUT_CSV_PATH, index=False)

print("-" * 50)
print(f"SUCCESS: Saved {len(final_df)} rows to {OUTPUT_CSV_PATH}")
print("-" * 50)

Loading raw dataset: /afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/train_labeled.csv
Executing bulletproof formatting (3 columns)...
--------------------------------------------------
SUCCESS: Saved 18461 rows to /afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/alsei/Thesis_IT_TicketClassification/data/nuuday_distillation_ready.csv
--------------------------------------------------


### Dataset Validation Checks (`validate_label`)

This script acts as a quality assurance filter for the processed dataset, ensuring the formatting strictly adheres to the Distill-Step-by-Step requirements. It evaluates the `label` column of every row for formatting failures and regex extraction fallbacks.

#### 1. The Reasoning Check
* **Validates:** Presence of the exact string `Reasoning:` at the start of a line.
* **Flags:** `Reasoning: N/A`. This indicates the extraction regex failed to find the reasoning block, preventing the student model from learning to output "N/A" instead of valid logic.

#### 2. The Tag Check
* **Validates:** Presence of the exact string `Tag:` at the start of a line.
* **Flags:** Missing tags. This is the critical classification target (e.g., `(1g Self service...)`); without it, the row lacks the ground-truth label required for training.

#### 3. The Scientific Confidence Check ($S^*$)
* **Validates:** Presence of the exact string `Scientific Confidence:` at the start of a line.
* **Flags:** `Scientific Confidence: 0.0`. Since the Teacher model rarely outputs exactly 0.0, this safely indicates that the score extraction regex failed for that specific row.

#### Output Categorization
Rows passing all checks are marked **"Valid"**. Failing rows are flagged with the specific error reason, allowing them to be cleanly filtered out or manually reviewed before training.

In [11]:
import re

# ==========================================
# 1. VALIDATION LOGIC
# ==========================================
def validate_label(label_text):
    label_text = str(label_text).strip()
    errors = []

    if not re.search(r"^Reasoning:", label_text, re.MULTILINE):
        errors.append("Missing 'Reasoning:' prefix")
    elif "Reasoning: N/A" in label_text:
        errors.append("Reasoning extracted as N/A")

    if not re.search(r"^Tag:", label_text, re.MULTILINE):
        errors.append("Missing 'Tag:' prefix")

    if not re.search(r"^Scientific Confidence:", label_text, re.MULTILINE):
        errors.append("Missing 'Scientific Confidence:' prefix")
    elif "Scientific Confidence: 0.0" in label_text:
        errors.append("Confidence is 0.0")

    return " | ".join(errors) if errors else "Valid"

# ==========================================
# 2. EXECUTION ON EXISTING DATAFRAME
# ==========================================
print("Executing in-memory validation checks...")

# Because we removed main() from Cell 1, final_df exists here!
final_df['qa_status'] = final_df['label'].apply(validate_label)

valid_df = final_df[final_df['qa_status'] == "Valid"]
error_df = final_df[final_df['qa_status'] != "Valid"]
total_rows = len(final_df)

# ==========================================
# 3. REPORTING
# ==========================================
print("-" * 50)
print("VALIDATION REPORT")
print("-" * 50)
print(f"Total rows:   {total_rows}")
print(f"Valid rows:   {len(valid_df)} ({(len(valid_df)/total_rows)*100:.2f}%)")
print(f"Invalid rows: {len(error_df)} ({(len(error_df)/total_rows)*100:.2f}%)")
print("-" * 50)

if not error_df.empty:
    print("Error Summary:")
    error_counts = error_df['qa_status'].value_counts()
    for error_type, count in error_counts.items():
        print(f"  - {error_type}: {count}")
    
    print("-" * 50)
    print("Detailed Error List (Top 20):")
    for index, row in error_df.head(20).iterrows():
        print(f"Row {index:05d} | {row['qa_status']}")
    print("-" * 50)
else:
    print("Validation complete. Zero errors detected.")

# Clean up the temporary column
final_df = final_df.drop(columns=['qa_status'])

Executing in-memory validation checks...
--------------------------------------------------
VALIDATION REPORT
--------------------------------------------------
Total rows:   18461
Valid rows:   18461 (100.00%)
Invalid rows: 0 (0.00%)
--------------------------------------------------
Validation complete. Zero errors detected.
